# Day 3: Dynamic Pages, Selenium, Debugging, and Robust Scraping

Some websites do not put the visible data into the first HTML response. They load content later with JavaScript, after the browser has opened the page.

This notebook starts with a short Wikipedia comparison, because Day 1 used the MediaWiki API and Day 2 used static Wikipedia HTML scraping. Then it moves to a JavaScript-rendered page where browser automation is actually needed.

By the end, you should understand:

- how browser automation differs from API access and static scraping
- why `requests` may miss JavaScript-rendered content
- how Selenium observes a rendered browser DOM
- how wait conditions shape collected data
- how scrolling and infinite-scroll stopping rules work
- why screenshots and rendered HTML are raw evidence
- how to debug empty or broken scrapers
- what bot-compliance boundaries should not be crossed


## Packages Used in This Notebook

The code chunk below loads the packages used in this notebook. The comments explain what each package contributes to the workflow.


In [27]:
# time lets us pause after browser actions such as scrolling, clicking, or typing.
# These pauses make automated browser actions visible during live teaching.
import time

# Path helps create output folders and file paths, such as data/raw or data/processed.
from pathlib import Path

# quote turns an HTML string into a safe data: URL so Selenium can open a tiny
# local teaching page without needing a separate web server.
from urllib.parse import quote

# pandas turns extracted browser records into tables and saves processed outputs.
import pandas as pd

# requests fetches the initial server HTML without opening a browser or running
# JavaScript. We use it first to diagnose whether Selenium is actually needed.
import requests

# BeautifulSoup parses initial HTML so we can test what static scraping can see
# before using a browser.
from bs4 import BeautifulSoup

# webdriver starts and controls a real browser from Python.
from selenium import webdriver

# Options stores Chrome settings such as headless mode, window size, and User-Agent.
from selenium.webdriver.chrome.options import Options

# By tells Selenium how to locate page elements, for example by CSS selector.
from selenium.webdriver.common.by import By

# Keys lets Selenium send keyboard keys such as Enter to an input field.
from selenium.webdriver.common.keys import Keys

# expected_conditions provides common wait rules, such as waiting until an element
# exists or until text appears on the page. We abbreviate it as EC below.
from selenium.webdriver.support import expected_conditions as EC

# WebDriverWait repeatedly checks a condition until it succeeds or times out.
# This is better than assuming the page is ready immediately.
from selenium.webdriver.support.ui import WebDriverWait


## Core Concepts: HTML, Rendered Page, DOM, and JavaScript

Before using Selenium, we need four basic terms.

**HTML** is the source text of a web page. It contains tags such as `<h1>`, `<p>`, `<a>`, `<div>`, and `<button>`. When we use `requests.get(url).text`, we get the initial HTML that the server sends back.

**Rendered page** means the page after the browser has turned HTML, CSS, and JavaScript into the visible page that a person sees and interacts with. Rendering is the browser’s work: it reads the source, applies styling, runs scripts, lays out the page, and displays it.

**DOM** stands for **Document Object Model**. It is the browser’s live tree representation of the page. The DOM starts from the HTML, but it can change after the page loads. JavaScript can add, remove, or modify DOM elements.

**JavaScript** is code that runs in the browser. It can make a page interactive: open menus, respond to clicks, submit forms, load more records, update search results, or create content after the first HTML response has arrived.

This is why some pages work with `requests` + `BeautifulSoup`, while others need a browser:

| Page type | What happens | Scraping implication |
|---|---|---|
| Mostly static page | Server sends HTML that already contains the content | `requests` + `BeautifulSoup` is often enough |
| JavaScript-rendered page | Server sends initial HTML, then browser JavaScript creates or changes content | Selenium may be needed to inspect the rendered DOM |

A useful diagnostic question is:

> Is the data present as normal HTML elements in the initial response, or only after the browser renders the page?

If the data is already in the initial HTML, use the simpler static route. If the data only appears after JavaScript runs, browser automation may be appropriate.


## Same Source, Third Access Route: Wikipedia in a Browser

First we open the same Wikipedia page in an automated browser.

This is useful for comparison, not because Wikipedia requires Selenium for basic article text. The API and static HTML route are usually better for this source. Here, Selenium helps us see what browser automation adds:

- rendered page state
- browser title and URL
- screenshots
- waits for visible page elements
- rendered HTML as browser evidence


In [28]:
# A User-Agent identifies this teaching script to the server.
# It is request metadata; it does not grant permission or bypass restrictions.
USER_AGENT = "methodsNET-VLOP-course/1.0 dynamic walkthrough"

# We use the same Wikipedia page as in the API/static scraping examples so
# you can compare three access routes: API, static HTML, and browser DOM.
WIKIPEDIA_URL = "https://en.wikipedia.org/wiki/Digital_Services_Act"

# Output folders mirror the rest of the course structure:
# raw = evidence close to what the source returned;
# processed = tables we create from that evidence;
# reports = diagnostics, counts, and provenance-like notes.
outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
reports_dir = outdir / "reports"

# mkdir(..., exist_ok=True) creates folders if needed and does nothing if they
# already exist. parents=True also creates missing parent folders.
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)


## 1. Browser Automation Where It Is Possible but Not Necessary

This function opens Wikipedia in Chrome, waits for the article heading, extracts a few simple fields, and saves a screenshot.

The point is to compare access routes:

- API: structured JSON from MediaWiki
- static scraping: HTML from `requests`
- browser automation: rendered DOM and screenshot from a real browser

For this particular page, browser automation is overkill. That is an important methodological conclusion.


In [29]:
def open_wikipedia_with_selenium(url, *, headless=False, pause_seconds=8):
    """Open a Wikipedia page in Chrome and collect rendered-page evidence."""

    # Selenium imports live inside the function so the notebook can still be read
    # before browser dependencies are installed. If Selenium is missing, the
    # error appears only when this browser-specific function is run.
    import time
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait

    # Options stores browser settings before Chrome is started.
    options = Options()
    # Add the same User-Agent idea as in requests-based scripts, but here it is
    # attached to the automated Chrome browser.
    options.add_argument(f"--user-agent={USER_AGENT}")

    # headless=False opens a visible browser window, which is better for teaching.
    # For automated scripts later, set headless=True so Chrome runs in the background.
    if headless:
        options.add_argument("--headless=new")

    # webdriver.Chrome(...) starts a Chrome browser controlled by Python.
    # The driver object is the handle we use to open pages, find elements, click,
    # type, scroll, take screenshots, and read the rendered DOM.
    driver = webdriver.Chrome(options=options)

    try:
        # driver.get(...) is the browser equivalent of entering a URL in the
        # address bar and loading the page.
        driver.get(url)

        # WebDriverWait repeatedly checks a condition until it is true or the
        # timeout is reached. This is better than immediately selecting elements,
        # because JavaScript-rendered pages may not be ready right away.
        wait = WebDriverWait(driver, 15)
        # presence_of_element_located waits until at least one h1 exists in the
        # rendered DOM. The tuple says: find by CSS selector, using selector h1.
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))

        # driver.title reads the browser tab title after the page has loaded.
        title = driver.title
        # find_element returns the first matching element. .text gives the
        # visible text as the browser sees it.
        heading = driver.find_element(By.CSS_SELECTOR, "h1").text
        # find_elements returns a list of all matching elements. len(...) counts
        # how many paragraph/link elements the browser currently sees.
        paragraph_count = len(driver.find_elements(By.CSS_SELECTOR, ".mw-parser-output p"))
        link_count = len(driver.find_elements(By.CSS_SELECTOR, ".mw-parser-output a[href]"))

        # page_source is the rendered DOM as seen by the browser after loading.
        # This can differ from requests.get(...).text when JavaScript modifies the page.
        rendered_html = driver.page_source
        # A screenshot is visual evidence of the rendered page state. It is useful
        # for checking cookie banners, empty states, blocked pages, or layout changes.
        screenshot = driver.get_screenshot_as_png()

        # In teaching mode, keep the browser open briefly so you can see it.
        # Set pause_seconds=0 if you do not want a pause.
        if pause_seconds > 0:
            time.sleep(pause_seconds)

    finally:
        # Always close the browser, even if extraction fails. Otherwise Chrome
        # windows/processes can remain open in the background.
        driver.quit()

    # Return a dictionary so later cells can save the evidence and diagnostics.
    return {
        "url": url,
        "browser_title": title,
        "heading": heading,
        "paragraph_count": paragraph_count,
        "link_count": link_count,
        "rendered_html": rendered_html,
        "screenshot": screenshot,
    }


# For the live classroom demo, headless=False makes Chrome visible.
# pause_seconds keeps the browser open long enough for you to see it.
wiki_browser = open_wikipedia_with_selenium(WIKIPEDIA_URL, headless=False, pause_seconds=8)

print("Browser title:", wiki_browser["browser_title"])
print("Heading:", wiki_browser["heading"])
print("Paragraph count:", wiki_browser["paragraph_count"])
print("Link count:", wiki_browser["link_count"])


Browser title: Digital Services Act - Wikipedia
Heading: Digital Services Act
Paragraph count: 54
Link count: 1024


## 2. Visible Browser Actions on Wikipedia

Before moving to a JavaScript-rendered scraping target, use Wikipedia to see what Selenium can do in a browser you already understand.

This example is not here because Wikipedia needs Selenium for article text. It usually does not. The point is to demonstrate browser actions on a familiar real page:

- open a URL;
- wait for an element;
- scroll gradually;
- click a link;
- go back;
- type into a search field;
- read the browser URL and title.

This makes Selenium visible as browser control, not just as an extraction library.


In [ ]:
def demo_wikipedia_browser_actions(url, *, pause_seconds=1.2):
    """Show visible Selenium actions on a familiar Wikipedia page."""

    # Wikipedia's own JavaScript can replace elements while the page is open.
    # If Selenium keeps an old element reference after that replacement, Selenium
    # raises a StaleElementReferenceException. We import that exception so we can
    # retry by finding the element again.
    from selenium.common.exceptions import StaleElementReferenceException

    def visible_search_box():
        """Find the currently visible and usable Wikipedia search box."""

        # There can be more than one search input in the page source, depending
        # on Wikipedia's layout. We only want the one that is displayed and enabled.
        search_boxes = driver.find_elements(By.CSS_SELECTOR, "input[name='search']")
        for box in search_boxes:
            if box.is_displayed() and box.is_enabled():
                return box
        return None

    def type_slowly_into_search(text):
        """Type into the search field while re-finding it if Wikipedia updates it."""

        # Wait until at least one search input exists.
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='search']")))

        box = visible_search_box()
        if box is None:
            raise RuntimeError("Could not find a visible Wikipedia search box.")

        box.click()
        box.clear()

        for character in text:
            # Re-find the visible input for each character. This is slower, but
            # robust for teaching because autocomplete/search JavaScript may
            # replace the input element while we type.
            for attempt in range(3):
                try:
                    box = visible_search_box()
                    if box is None:
                        raise RuntimeError("Could not find a visible Wikipedia search box.")
                    box.send_keys(character)
                    break
                except StaleElementReferenceException:
                    # The page replaced the element between finding and typing.
                    # Wait a moment and try again with a fresh element reference.
                    time.sleep(0.1)
            time.sleep(0.08)

        return visible_search_box()

    options = Options()
    options.add_argument(f"--user-agent={USER_AGENT}")
    options.add_argument("--window-size=1200,850")

    # Keep this browser visible because this is a teaching demonstration.
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 15)

    try:
        # Open the Wikipedia article.
        driver.get(url)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
        time.sleep(pause_seconds)

        # Read the visible page heading.
        heading = driver.find_element(By.CSS_SELECTOR, "h1").text
        print("Started on:", heading)

        # Scroll down gradually so you can see the page move.
        for step in range(5):
            driver.execute_script("window.scrollBy(0, 350);")
            time.sleep(0.4)

        # Scroll back to the top before clicking a link in the lead/content area.
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(pause_seconds)

        # Click the first internal Wikipedia link inside the article content.
        # This demonstrates that Selenium can navigate by clicking page elements.
        first_article_link = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".mw-parser-output p a[href^='/wiki/']"))
        )
        clicked_text = first_article_link.text
        print("Clicking link:", clicked_text)
        first_article_link.click()
        time.sleep(2)
        print("After click, browser title:", driver.title)

        # Go back to the original article, like clicking the browser Back button.
        driver.back()
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
        time.sleep(pause_seconds)

        # Type into Wikipedia's search field. The search box may be controlled by
        # JavaScript/autocomplete, so the helper function re-finds it while typing.
        search_box = type_slowly_into_search("Digital Markets Act")

        time.sleep(pause_seconds)

        # Press Enter to submit the search/navigation. Re-find the box once more
        # before pressing Enter in case the page updated it during typing.
        search_box = visible_search_box()
        if search_box is None:
            raise RuntimeError("Could not find a visible Wikipedia search box before pressing Enter.")
        search_box.send_keys(Keys.ENTER)
        time.sleep(2)

        print("After search, current URL:", driver.current_url)
        print("After search, browser title:", driver.title)

        # Keep the final page visible briefly before closing.
        time.sleep(3)

    finally:
        # Close the visible browser after the demonstration.
        driver.quit()


# Run this once as a live demo. It opens a visible browser window.
demo_wikipedia_browser_actions(WIKIPEDIA_URL)


## 2. Save Wikipedia Browser Evidence

For browser automation, raw evidence should include not only extracted tables but also rendered HTML and screenshots. These help diagnose cookie banners, broken rendering, empty states, or unexpected page versions.


In [12]:
# Define output paths for the browser-rendered Wikipedia evidence.
wiki_rendered_path = raw_dir / "notebook_wikipedia_dsa_selenium_rendered.html"
wiki_screenshot_path = raw_dir / "notebook_wikipedia_dsa_selenium.png"
wiki_diagnostics_path = reports_dir / "notebook_wikipedia_dsa_selenium_diagnostics.json"

# Save the rendered DOM as raw evidence. This is what Selenium saw after Chrome
# loaded the page, not necessarily the same as the initial server HTML.
wiki_rendered_path.write_text(wiki_browser["rendered_html"], encoding="utf-8")
# The screenshot is binary PNG data, so we save it with write_bytes().
wiki_screenshot_path.write_bytes(wiki_browser["screenshot"])

# pd.Series(...).to_json(...) is a quick way to save one dictionary-like record
# with small diagnostics about the browser run.
pd.Series(
    {
        "url": wiki_browser["url"],
        "browser_title": wiki_browser["browser_title"],
        "heading": wiki_browser["heading"],
        "paragraph_count": wiki_browser["paragraph_count"],
        "link_count": wiki_browser["link_count"],
        "method_note": "Browser automation works here, but API/static scraping are usually better for this Wikipedia task.",
    }
).to_json(wiki_diagnostics_path, indent=2)

print(wiki_rendered_path)
print(wiki_screenshot_path)
print(wiki_diagnostics_path)


data/raw/notebook_wikipedia_dsa_selenium_rendered.html
data/raw/notebook_wikipedia_dsa_selenium.png
data/reports/notebook_wikipedia_dsa_selenium_diagnostics.json


## Main Dynamic Example: Quotes to Scrape JS

Now we switch to a page where browser automation is more clearly justified.

`quotes.toscrape.com/js/` is a teaching page where the visible quotes are rendered by JavaScript. That makes it useful for comparing what `requests` sees with what a browser sees.


In [30]:
URL = "https://quotes.toscrape.com/js/"


## 3. Fetch the JS Page with `requests` First

Always start by checking the simple route.

`requests.get()` asks the server for the initial HTML document, but it does not execute JavaScript, click buttons, scroll, or wait for client-side rendering.

Important distinction:

- a word or quote may appear somewhere in the raw HTML source, for example inside a `<script>` block;
- that does **not** mean the content is already present as normal HTML records that BeautifulSoup can extract with selectors such as `.quote`.

So the better diagnostic is not only “does the raw text contain a phrase?” but also:

> Can BeautifulSoup find the record containers we would normally scrape?


In [31]:
# requests.get() sends a normal HTTP GET request.
static_response = requests.get(URL, headers={"User-Agent": USER_AGENT}, timeout=30)

# raise_for_status() turns HTTP error statuses into visible Python errors.
static_response.raise_for_status()

# response.text is the initial HTML returned by the server.
# This is source HTML, not the browser-rendered DOM after JavaScript runs.
static_html = static_response.text

# Parse the initial HTML with BeautifulSoup to test what static scraping can see.
# If this finds zero .quote records, the usual BeautifulSoup workflow cannot yet
# extract the visible quotes as repeated HTML containers.
from bs4 import BeautifulSoup

static_soup = BeautifulSoup(static_html, "html.parser")
static_quote_blocks = static_soup.select(".quote")

print("Static HTML characters:", len(static_html))
print("Does raw HTML source contain quote text?", "The world as we have created it" in static_html)
print("Number of .quote records BeautifulSoup can select:", len(static_quote_blocks))


Static HTML characters: 5806
Does raw HTML source contain quote text? True
Number of .quote records BeautifulSoup can select: 0


## 4. Diagnose: Rendering Problem or Access Problem?

A missing record can mean different things.

It might be a rendering problem:

- the server returned HTML;
- the target text may even appear somewhere in the source, for example inside JavaScript;
- but the repeated record containers are not present as normal HTML elements yet;
- the browser later runs JavaScript and creates the visible records;
- Selenium may be appropriate.

Or it might be an access/compliance problem:

- `403 Forbidden`;
- `429 Too Many Requests`;
- login wall;
- paywall;
- CAPTCHA / prove-you-are-human page;
- consent wall that changes access conditions.

Browser automation is for observing rendered public content. It should not be used to bypass access controls.


In [32]:
# This diagnostic dictionary asks: did the normal requests-based route already
# return extractable HTML records, or do we need a rendered browser DOM?
diagnosis = {
    # HTTP status code from the non-browser request.
    "status_code": static_response.status_code,
    # Content type tells us what kind of response the server says it returned.
    "content_type": static_response.headers.get("content-type"),
    # Length is a rough sanity check: tiny HTML may indicate an error or empty shell.
    "static_html_characters": len(static_html),
    # This only checks whether the phrase occurs anywhere in the source text.
    # It may be inside JavaScript, so this alone does not prove static scraping works.
    "contains_expected_quote_text_anywhere_in_source": "The world as we have created it" in static_html,
    # This is the stronger static-scraping test: can BeautifulSoup select the
    # repeated quote containers from the initial HTML?
    "static_quote_records_found_by_beautifulsoup": len(static_quote_blocks),
    # These are quick warning signs. They are not perfect detection rules, but
    # they remind us to check for access barriers before scraping.
    "contains_captcha_word": "captcha" in static_html.lower(),
    "contains_login_word": "login" in static_html.lower(),
}

diagnosis


{'status_code': 200,
 'content_type': 'text/html; charset=utf-8',
 'static_html_characters': 5806,
 'contains_expected_quote_text_anywhere_in_source': True,
 'static_quote_records_found_by_beautifulsoup': 0,
 'contains_captcha_word': False,
 'contains_login_word': True}

## 5. What Selenium Is For

Selenium is a browser-automation tool. It lets Python control a real browser in roughly the way a user would:

- open a URL;
- wait until an element appears;
- read visible text;
- read attributes such as `href`, `src`, or `aria-label`;
- click buttons or links;
- type into forms;
- submit a form;
- scroll;
- take screenshots;
- save the rendered HTML.

Selenium is useful when the simple static route is insufficient, for example when:

- content is added by JavaScript after the first HTML response;
- a page requires clicking a tab, button, menu, or “load more” control to reveal public content;
- the page uses infinite scroll;
- you need a screenshot or rendered-page evidence;
- you need to test whether selectors work in the browser DOM, not only in saved HTML.

Selenium is usually **not** the best first choice when an API or simple static HTML request gives the needed data more cleanly. It is slower, more fragile, and easier to misuse.


## 6. Common Selenium Operations

The examples below are reference patterns. They are not all needed for the quotes example, but they are important to recognize when reading or adapting browser-automation scripts.


In [ ]:
# This cell is a reference sheet, not a workflow to run as-is.
# It assumes a Selenium driver already exists:
# driver = webdriver.Chrome(options=options)

# Open a page.
# driver.get("https://example.com")

# Find one element by CSS selector.
# heading = driver.find_element("css selector", "h1")
# print(heading.text)

# Find many elements by CSS selector.
# links = driver.find_elements("css selector", "a[href]")
# for link in links:
#     print(link.text, link.get_attribute("href"))

# Wait until an element exists before trying to use it.
# wait = WebDriverWait(driver, 15)
# wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

# Click a button or link.
# button = driver.find_element("css selector", "button.load-more")
# button.click()

# Type into a search box or form field.
# search_box = driver.find_element("css selector", "input[name='q']")
# search_box.clear()
# search_box.send_keys("digital services act")

# Submit by pressing Enter.
# from selenium.webdriver.common.keys import Keys
# search_box.send_keys(Keys.ENTER)

# Take a screenshot.
# driver.get_screenshot_as_file("page_state.png")

# Save rendered HTML.
# rendered_html = driver.page_source


### Runnable Mini Demo: Clicking, Typing, Waiting, and Scrolling

The reference sheet above shows common Selenium commands, but for teaching it helps to watch them happen.

This mini demo uses a tiny HTML page created inside the notebook. That keeps the example stable: it does not depend on an external website, and it does not require logging into anything.

The demo shows how Selenium can:

- open a page in a visible browser window;
- type into an input field;
- click buttons;
- wait until JavaScript changes the page;
- read visible text;
- read attributes;
- scroll;
- save a screenshot.

This is the same interaction vocabulary you would use on real public pages when clicking tabs, opening menus, pressing “load more,” or revealing dynamically loaded content.


In [33]:
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# Change this if the demo is too slow or too fast in the classroom.
DEMO_PAUSE = 1.2

# Helper: type one character at a time so you can see Selenium typing.
def slow_type(element, text, delay=0.12):
    for character in text:
        element.send_keys(character)
        time.sleep(delay)

# Helper: scroll gradually so you can see the movement.
def slow_scroll(driver, pixels=900, step=150, delay=0.25):
    scrolled = 0
    while scrolled < pixels:
        driver.execute_script("window.scrollBy(0, arguments[0]);", step)
        scrolled += step
        time.sleep(delay)

# A tiny local HTML page for teaching browser interaction.
# It contains an input field, buttons, dynamic text, links, and enough vertical
# space to make scrolling visible.
demo_html = """
<!doctype html>
<html>
<head>
  <title>Selenium mini demo</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 30px; line-height: 1.4; }
    h1 { margin-bottom: 8px; }
    .note { color: #555; margin-bottom: 24px; }
    .panel { border: 2px solid #777; padding: 18px; margin-bottom: 22px; max-width: 760px; }
    label { display: block; font-weight: bold; margin-bottom: 6px; }
    input { font-size: 20px; padding: 8px; width: 420px; }
    button { font-size: 18px; padding: 9px 14px; margin-left: 6px; cursor: pointer; }
    .result { padding: 10px; background: #eef; margin-top: 12px; font-weight: bold; }
    .record { margin: 8px 0; padding: 6px; background: #f7f7f7; }
    .spacer { height: 950px; background: linear-gradient(#fff, #eee); padding-top: 30px; color: #555; }
    #bottom-section { border-top: 4px solid #444; padding-top: 16px; }
  </style>
</head>
<body>
  <h1>Selenium Mini Demo Page</h1>
  <p class="note">Watch the browser: Python will type, click, wait, add records, and scroll.</p>

  <div class="panel">
    <label for="query">Search term</label>
    <input id="query" name="q" placeholder="type something here">
    <button id="search-button">Search</button>
    <p id="search-result" class="result">No search yet.</p>
  </div>

  <div class="panel">
    <button id="load-more" data-batch="1">Load more records</button>
    <ul id="records">
      <li class="record" data-id="r1"><a href="/record/1">Record 1</a></li>
    </ul>
  </div>

  <div class="spacer">There is a lower section below. Selenium will scroll to it slowly.</div>

  <h2 id="bottom-section">Bottom Section</h2>
  <p>This section is only visible after scrolling.</p>

  <script>
    document.querySelector('#search-button').addEventListener('click', function() {
      const query = document.querySelector('#query').value;
      document.querySelector('#search-result').textContent = 'You searched for: ' + query;
    });

    document.querySelector('#query').addEventListener('keydown', function(event) {
      if (event.key === 'Enter') {
        document.querySelector('#search-button').click();
      }
    });

    document.querySelector('#load-more').addEventListener('click', function() {
      const records = document.querySelector('#records');
      const nextNumber = records.querySelectorAll('.record').length + 1;
      const item = document.createElement('li');
      item.className = 'record';
      item.setAttribute('data-id', 'r' + nextNumber);
      item.innerHTML = '<a href="/record/' + nextNumber + '">Record ' + nextNumber + '</a>';
      records.appendChild(item);
    });
  </script>
</body>
</html>
"""

# A data: URL lets Selenium open our teaching HTML as if it were a page.
demo_url = "data:text/html;charset=utf-8," + quote(demo_html)

options = Options()
options.add_argument(f"--user-agent={USER_AGENT}")
options.add_argument("--window-size=1100,850")

# Use a visible browser for teaching. Change to True later for automation.
headless = False
if headless:
    options.add_argument("--headless=new")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

try:
    # 1. Open the demo page.
    driver.get(demo_url)
    time.sleep(DEMO_PAUSE)

    # 2. Find an input field, clear it, click it, and type into it slowly.
    query_box = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#query")))
    query_box.click()
    time.sleep(DEMO_PAUSE)
    query_box.clear()
    slow_type(query_box, "digital services act")
    time.sleep(DEMO_PAUSE)

    # 3. Submit the input by pressing Enter.
    query_box.send_keys(Keys.ENTER)
    time.sleep(DEMO_PAUSE)

    # 4. Wait until the page text changes after JavaScript runs.
    wait.until(EC.text_to_be_present_in_element((By.CSS_SELECTOR, "#search-result"), "digital services act"))
    print(driver.find_element(By.CSS_SELECTOR, "#search-result").text)

    # 5. Click a button that adds records to the page. Pause between clicks so
    # you can see the list grow.
    load_more_button = driver.find_element(By.CSS_SELECTOR, "#load-more")
    load_more_button.click()
    time.sleep(DEMO_PAUSE)
    load_more_button.click()
    time.sleep(DEMO_PAUSE)

    # 6. Read visible text and attributes from the newly added records.
    records = driver.find_elements(By.CSS_SELECTOR, ".record")
    for record in records:
        link = record.find_element(By.CSS_SELECTOR, "a[href]")
        print(record.get_attribute("data-id"), link.text, link.get_attribute("href"))

    # 7. Scroll slowly so the movement is visible.
    slow_scroll(driver, pixels=1000, step=125, delay=0.2)
    bottom = driver.find_element(By.CSS_SELECTOR, "#bottom-section")
    print("Reached:", bottom.text)
    time.sleep(2)

    # 8. Save a screenshot as evidence of the browser state.
    screenshot_path = raw_dir / "notebook_selenium_mini_demo.png"
    driver.get_screenshot_as_file(str(screenshot_path))
    print("Saved screenshot:", screenshot_path)

finally:
    # Close the browser even if something fails.
    driver.quit()


You searched for: digital services act
r1 Record 1 /record/1
r2 Record 2 /record/2
r3 Record 3 /record/3
Reached: Bottom Section
Saved screenshot: data/raw/notebook_selenium_mini_demo.png


### In-Class Exercise: Automate a Tiny Website You Can Inspect

In this exercise, you will work with a small HTML page created inside this notebook. The notebook turns the HTML into a local page that Selenium can open.

This is useful because the page is fully transparent: you can inspect the HTML, render it in a browser, and then automate interactions with it. There is no platform policy issue, no login, no CAPTCHA, and no hidden website behavior.

The exercise connects three things:

1. **HTML source:** what the page is made of;
2. **Rendered page:** what the browser shows;
3. **Selenium actions:** how Python clicks, types, waits, scrolls, and extracts from the rendered DOM.

Your tasks:

1. Open the local page with Selenium.
2. Type a topic into the search field.
3. Click the search button or press Enter.
4. Wait until the result text changes.
5. Click the “Add card” button several times.
6. Extract all card titles and `data-id` attributes.
7. Scroll to the bottom section.
8. Save a screenshot.

First run the starter code as written. Then change at least one thing: the search term, number of clicks, selector, or extracted fields.


In [ ]:
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

exercise_html = """
<!doctype html>
<html>
<head>
  <title>Student Selenium Exercise</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 32px; line-height: 1.4; }
    .toolbar { border: 2px solid #555; padding: 16px; margin-bottom: 20px; max-width: 820px; }
    input { font-size: 18px; padding: 7px; width: 360px; }
    button { font-size: 17px; padding: 8px 12px; margin: 6px; cursor: pointer; }
    #status { background: #eef; padding: 10px; font-weight: bold; }
    .card { border: 1px solid #888; padding: 12px; margin: 10px 0; max-width: 650px; background: #fafafa; }
    .card a { font-weight: bold; }
    .long-space { height: 850px; background: linear-gradient(#fff, #eee); padding-top: 30px; color: #555; }
    #bottom { border-top: 4px solid #444; padding-top: 16px; }
  </style>
</head>
<body>
  <h1>Student Selenium Exercise Page</h1>
  <p>This fake page behaves like a tiny interactive website.</p>

  <div class="toolbar">
    <label for="topic">Topic</label><br>
    <input id="topic" name="topic" placeholder="enter a topic">
    <button id="search">Search</button>
    <button id="add-card">Add card</button>
    <p id="status">No topic searched yet.</p>
  </div>

  <section id="cards" aria-label="results cards">
    <article class="card" data-id="card-1" data-topic="platforms">
      <a href="/items/1">Platform governance primer</a>
      <p>First visible card.</p>
    </article>
  </section>

  <div class="long-space">Scroll down after adding cards.</div>

  <h2 id="bottom">Bottom of the Exercise Page</h2>
  <p>This is where you should scroll before taking a screenshot.</p>

  <script>
    let cardCounter = 1;

    document.querySelector('#search').addEventListener('click', function() {
      const topic = document.querySelector('#topic').value;
      document.querySelector('#status').textContent = 'Current topic: ' + topic;
    });

    document.querySelector('#topic').addEventListener('keydown', function(event) {
      if (event.key === 'Enter') {
        document.querySelector('#search').click();
      }
    });

    document.querySelector('#add-card').addEventListener('click', function() {
      cardCounter = cardCounter + 1;
      const topic = document.querySelector('#topic').value || 'untitled';
      const card = document.createElement('article');
      card.className = 'card';
      card.setAttribute('data-id', 'card-' + cardCounter);
      card.setAttribute('data-topic', topic);
      card.innerHTML = '<a href="/items/' + cardCounter + '">Generated card ' + cardCounter + '</a>' +
                       '<p>Generated for topic: ' + topic + '</p>';
      document.querySelector('#cards').appendChild(card);
    });
  </script>
</body>
</html>
"""

# Turn the HTML string into a local data URL. This is a quick way to make our
# own small page without needing a web server.
exercise_url = "data:text/html;charset=utf-8," + quote(exercise_html)

options = Options()
options.add_argument("--window-size=1100,850")

# Keep the browser visible for the exercise.
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

try:
    # TODO 1: Open the local exercise page.
    driver.get(exercise_url)
    time.sleep(1)

    # TODO 2: Select the input field by its id and type a topic.
    topic_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#topic")))
    topic_input.click()
    topic_input.send_keys("platform accountability")
    time.sleep(1)

    # TODO 3: Submit the topic. Try either pressing Enter or clicking Search.
    topic_input.send_keys(Keys.ENTER)

    # TODO 4: Wait until the status text contains the topic.
    wait.until(EC.text_to_be_present_in_element((By.CSS_SELECTOR, "#status"), "platform accountability"))
    print(driver.find_element(By.CSS_SELECTOR, "#status").text)

    # TODO 5: Click the Add card button three times.
    add_button = driver.find_element(By.CSS_SELECTOR, "#add-card")
    for click_number in range(3):
        add_button.click()
        time.sleep(0.7)

    # TODO 6: Extract visible card titles and hidden data-id attributes.
    cards = driver.find_elements(By.CSS_SELECTOR, ".card")
    for card in cards:
        card_id = card.get_attribute("data-id")
        card_topic = card.get_attribute("data-topic")
        card_title = card.find_element(By.CSS_SELECTOR, "a").text
        print(card_id, card_topic, card_title)

    # TODO 7: Scroll to the bottom section.
    bottom = driver.find_element(By.CSS_SELECTOR, "#bottom")
    driver.execute_script("arguments[0].scrollIntoView();", bottom)
    time.sleep(2)

    # TODO 8: Save a screenshot.
    screenshot_path = raw_dir / "notebook_selenium_exercise.png"
    driver.get_screenshot_as_file(str(screenshot_path))
    print("Saved screenshot:", screenshot_path)

finally:
    # Close Chrome after the exercise run.
    driver.quit()


## 7. Login and Form Automation: Important Boundary

Selenium can type into username and password fields, but this should be treated carefully.

Acceptable teaching/research cases might include:

- logging into your own account when the platform permits automated access;
- using a test account in a controlled teaching environment;
- automating an internal tool with permission;
- documenting a workflow where the credentials and access rights are already legitimate.

Do **not** use Selenium to bypass paywalls, CAPTCHAs, “prove you are human” checks, account restrictions, rate limits, or platform access controls.

Never hard-code passwords in a script. If credentials are needed, read them from environment variables or enter them interactively.


In [ ]:
# Example pattern for a permitted/test login form.
# Do not run this against real platforms unless you have permission and the site's rules allow it.

# import os
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.support.ui import WebDriverWait

# username = os.getenv("COURSE_DEMO_USERNAME")
# password = os.getenv("COURSE_DEMO_PASSWORD")

# if username is None or password is None:
#     raise RuntimeError("Set COURSE_DEMO_USERNAME and COURSE_DEMO_PASSWORD first.")

# driver.get("https://example.com/login")
# wait = WebDriverWait(driver, 15)

# username_box = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='username']")))
# password_box = driver.find_element(By.CSS_SELECTOR, "input[name='password']")

# username_box.clear()
# username_box.send_keys(username)
# password_box.clear()
# password_box.send_keys(password)

# submit_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
# submit_button.click()

# Wait for evidence that login succeeded, such as a known page element.
# wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".account-dashboard")))


## 8. Browser Helper Functions

Selenium controls a real browser from Python.

We define helpers for three reasons:

1. keep the browser workflow readable
2. make diagnostics reproducible
3. make scrolling use an explicit stopping rule


In [ ]:
def describe_browser_state(driver):
    """Return small diagnostics about what the browser currently sees."""

    # This function is a debugging helper. It does not extract the final dataset;
    # it records browser state so we can understand what happened during the run.
    return {
        # current_url is useful because browser navigation or redirects may change
        # the final page compared with the URL we requested.
        "current_url": driver.current_url,
        # title is the browser tab title.
        "title": driver.title,
        # page_source is the current rendered DOM; its length is a rough sanity check.
        "html_characters": len(driver.page_source),
        # These counts check whether expected record/link elements are present.
        "quote_count": len(driver.find_elements("css selector", ".quote")),
        "link_count": len(driver.find_elements("css selector", "a[href]")),
    }


def scroll_until_stable(driver, *, record_selector, max_scrolls=5, wait_seconds=1.0):
    """Scroll until record count stops changing or max_scrolls is reached."""

    # counts stores the number of records after each scroll. This is a simple
    # monitoring trace: it tells us whether scrolling actually revealed more data.
    counts = []
    # Count records before any scrolling. This is the baseline.
    previous_count = len(driver.find_elements("css selector", record_selector))
    counts.append(previous_count)

    # range(1, max_scrolls + 1) produces 1, 2, ..., max_scrolls for readable logs.
    for scroll_number in range(1, max_scrolls + 1):
        # execute_script runs JavaScript in the browser. Here it scrolls to the
        # bottom of the page, which often triggers lazy loading or infinite scroll.
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Give the page time to load additional records after scrolling.
        time.sleep(wait_seconds)

        # Count records again after the scroll.
        current_count = len(driver.find_elements("css selector", record_selector))
        counts.append(current_count)

        print(f"Scroll {scroll_number}: {current_count} records")

        # Stop when scrolling no longer reveals additional records. This prevents
        # an infinite loop and creates a transparent stopping rule.
        if current_count == previous_count:
            break
        previous_count = current_count

    return counts


## 9. Selenium Collection Function

This function opens a browser, waits for rendered quote blocks, scrolls with a stopping rule, extracts records, and takes a screenshot.

Important methodological choices:

- `headless`: whether the browser is visible
- wait selector: here `.quote`
- scroll stopping rule: stop when record count stops increasing
- raw evidence: rendered HTML and screenshot


In [ ]:
def collect_rendered_page_with_selenium(url, *, headless=True, max_scrolls=2, wait_seconds=1.0):
    """Open a page in Chrome, wait for rendered content, and extract quote rows."""

    # Selenium imports live inside the function so the notebook can be read even
    # before Selenium/browser dependencies are installed.
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait

    # Options collects Chrome settings before the browser is launched.
    options = Options()
    # Identify the teaching script in the browser request metadata.
    options.add_argument(f"--user-agent={USER_AGENT}")

    # headless=True runs without a visible browser window.
    # During live teaching, set headless=False so you can watch the browser.
    if headless:
        options.add_argument("--headless=new")

    # Start the automated Chrome browser.
    driver = webdriver.Chrome(options=options)

    try:
        # driver.get() opens the page in the automated browser.
        driver.get(url)

        # Wait until at least one rendered quote block exists.
        # This is stronger than waiting for the body alone because it checks for
        # content that matters for our extraction.
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

        # Save diagnostics before scrolling so we can compare page state later.
        before_scroll = describe_browser_state(driver)

        # Scroll with a record-count stopping rule. This matters for pages where
        # more records appear only after scrolling.
        scroll_counts = scroll_until_stable(
            driver,
            record_selector=".quote",
            max_scrolls=max_scrolls,
            wait_seconds=wait_seconds,
        )

        # Save diagnostics after scrolling. If counts changed, scrolling mattered.
        after_scroll = describe_browser_state(driver)

        # page_source is rendered DOM after JavaScript execution.
        rendered_html = driver.page_source

        # Find all quote containers in the rendered browser DOM.
        quote_blocks = driver.find_elements(By.CSS_SELECTOR, ".quote")
        quotes = []

        for block in quote_blocks:
            # Search inside the current quote block so text, author, and tags stay
            # attached to the same record.
            quote_text = block.find_element(By.CSS_SELECTOR, ".text").text
            author = block.find_element(By.CSS_SELECTOR, ".author").text
            tags = [tag.text for tag in block.find_elements(By.CSS_SELECTOR, ".tag")]

            quotes.append({"quote": quote_text, "author": author, "tags": tags})

        # Screenshot is visual audit evidence of what the browser saw.
        screenshot = driver.get_screenshot_as_png()

    finally:
        # Always close automated browsers, including when an error occurs.
        driver.quit()

    # Keep diagnostics separate from data. They document how the collection ran.
    diagnostics = {
        "before_scroll": before_scroll,
        "after_scroll": after_scroll,
        "scroll_counts": scroll_counts,
    }

    return rendered_html, quotes, screenshot, diagnostics


## 10. Run the Browser Collection

For class, consider setting `headless=False` once so you can see the browser open and render.


In [ ]:
# Run the reusable browser collection function on the JavaScript-rendered page.
# headless=True is appropriate here because we already showed the visible browser
# demo above; this cell is closer to an automated collection workflow.
rendered_html, quotes, screenshot, diagnostics = collect_rendered_page_with_selenium(
    URL,
    headless=True,
    max_scrolls=2,
    wait_seconds=1.0,
)

print("Rendered HTML characters:", len(rendered_html))
print("Quotes extracted:", len(quotes))
print("Diagnostics:", diagnostics)

# Convert the list of quote dictionaries into a table for inspection.
pd.DataFrame(quotes).head()


## 11. Save JS-Page Evidence

For dynamic pages, a CSV alone is not enough.

Save:

- rendered HTML: what the browser saw after JavaScript
- screenshot: visual audit of page state
- processed CSV: extracted table
- diagnostics JSON: counts, scroll history, and other checks


In [ ]:
# Define output paths for each artifact produced by the browser workflow.
rendered_path = raw_dir / "notebook_rendered_quotes_selenium.html"
screenshot_path = raw_dir / "notebook_rendered_quotes_selenium.png"
quotes_path = processed_dir / "notebook_rendered_quotes_selenium.csv"
diagnostics_path = reports_dir / "notebook_rendered_quotes_selenium_diagnostics.json"

# Save rendered HTML as raw evidence.
rendered_path.write_text(rendered_html, encoding="utf-8")
# Save the screenshot as binary PNG evidence.
screenshot_path.write_bytes(screenshot)
# Save the processed quote table for analysis.
pd.DataFrame(quotes).to_csv(quotes_path, index=False)
# Save diagnostics so the collection process is documented, not only the result.
pd.Series(diagnostics).to_json(diagnostics_path, indent=2)

print(rendered_path)
print(screenshot_path)
print(quotes_path)
print(diagnostics_path)


## 12. Debugging Checklist

When a scraper returns zero records or strange records, debug in layers.


In [ ]:
debugging_checklist = [
    "Check response.status_code before parsing anything.",
    "Compare response.text with what the browser visibly shows.",
    "Save raw static HTML and rendered HTML for comparison.",
    "Inspect one record container before looping over all records.",
    "Use explicit waits for content-specific selectors, not only fixed sleep times.",
    "Save a screenshot to catch cookie banners, empty states, or blocked pages.",
    "Count records before and after scrolling; define a stopping rule.",
    "If selectors return zero, inspect whether markup changed.",
    "If you see login, paywall, CAPTCHA, or prove-you-are-human pages, stop and document.",
]

for item in debugging_checklist:
    print("-", item)


## 13. Anti-Fragile Patterns and Anti-Patterns

Anti-fragile does not mean impossible to break. It means the scraper is designed so breakage is visible, documented, and easier to repair.


In [ ]:
patterns = {
    "good_patterns": [
        "Wait for a content-specific selector.",
        "Save raw/rendered evidence and screenshots.",
        "Use small test runs before scaling.",
        "Keep selectors close to the record container.",
        "Log counts, URLs, status codes, and output paths.",
        "Stop when scroll counts no longer increase.",
        "Prefer official APIs or data downloads when available.",
    ],
    "anti_patterns": [
        "Using time.sleep() as the only waiting strategy.",
        "Looping forever on infinite scroll without a stopping rule.",
        "Ignoring empty outputs because the script did not crash.",
        "Treating a CAPTCHA or login wall as a technical obstacle to bypass.",
        "Collecting rapidly without rate limits or documentation.",
        "Saving only the final CSV and no raw evidence.",
    ],
}

for label, items in patterns.items():
    print(label)
    for item in items:
        print("-", item)


## 14. Bot-Compliance Decision Tree

This course teaches browser automation as a way to observe rendered public pages, not as a way to evade restrictions.


In [ ]:
bot_compliance_decision_tree = [
    "If content is missing because JavaScript renders it: browser automation may be appropriate.",
    "If a public API or bulk download exists: prefer that route when it fits the research question.",
    "If the site returns 429: pause and respect retry/rate-limit information.",
    "If the site returns 403: do not force access; check documentation and permissions.",
    "If the site shows CAPTCHA or human verification: stop; do not automate around it.",
    "If login, paywall, or terms restrict access: request permission or redesign the data source.",
    "Always document blocks, exclusions, and access limitations in provenance.",
]

for item in bot_compliance_decision_tree:
    print("-", item)


## 15. Where Playwright Fits

Selenium is a good starting point because it makes the idea of controlling a browser concrete.

Playwright is a newer browser-automation library with strong auto-waiting, browser contexts, network inspection, and tracing. For more complex modern web apps, Playwright can be easier to debug.

In this course:

- start with Selenium to understand browser automation
- mention Playwright as a modern alternative
- use the runnable Playwright script if you want to compare tools or show network-oriented debugging


## Exercise

Choose one debugging scenario and write what you would check:

1. `requests` returns HTML, but the target records are missing.
2. Selenium opens the page, but extracts zero records.
3. Scrolling does not increase the record count.
4. The screenshot shows a cookie/consent banner covering the page.
5. The page shows a CAPTCHA or login wall.

For each scenario, answer:

- Is this a rendering issue, selector issue, timing issue, or access/compliance boundary?
- What evidence would you save?
- What would you try next?
- What would you not do?

## Key Takeaway

Dynamic scraping is not just Selenium. It is diagnosis, waiting, evidence, stopping rules, debugging, and compliance.
